# 🎬 Video Chef — Text-to-Video (Wan 2.2)

Generate a short video from a text prompt using the open-source **Wan 2.2** family (Apache 2.0).

**Default model: `TI2V-5B`** — runs on a single **L4 (24 GB)** pay-as-you-go Colab GPU at 720p @ 24fps.
For higher quality, switch to `T2V-A14B` (requires A100 40GB).

| Model | Task | Min VRAM | Colab tier | Speed (5s @ 720p) |
|---|---|---|---|---|
| `ti2v-5B` | T2V + I2V | ~24 GB | L4 | ~9 min |
| `t2v-A14B` | T2V (MoE 27B/14B active) | ~40 GB with offload | A100 40GB | ~15–25 min |

> Runtime → Change runtime type → **L4** (or A100 if you picked A14B).


In [ ]:
# @title 0. Check GPU
!nvidia-smi

In [ ]:
# @title 1. Mount Google Drive (for weight caching across sessions)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/Wan2.2/outputs
print('Drive mounted. Weights root: /content/drive/MyDrive/Wan2.2/ (existing structure preserved)')

In [ ]:
# @title 2. Clone Wan 2.2 repo + install dependencies
%cd /content
![ -d Wan2.2 ] || git clone --depth 1 https://github.com/Wan-Video/Wan2.2.git
%cd /content/Wan2.2
# Install core dependencies
!pip install -q -e .
!pip install -q -r requirements.txt
# Specific fixes for missing NLP tools and HuggingFace
!pip install -q ftfy regex "huggingface_hub[cli]"
print('Setup done. Modules successfully installed.')

In [ ]:
# @title 3. Download model weights (shared across notebooks)
MODEL = "TI2V-5B"  # @param ["TI2V-5B", "T2V-A14B"]
REPO_ID = f"Wan-AI/Wan2.2-{MODEL}"
CKPT_DIR = f"/content/drive/MyDrive/Wan2.2/Wan2.2-{MODEL}"

from huggingface_hub import snapshot_download
import os, shutil

os.makedirs(CKPT_DIR, exist_ok=True)

# Check disk space to prevent crash
_, _, free = shutil.disk_usage("/")
print(f"Storage check: {free // (2**30)} GB free on local disk.")

if os.path.exists(os.path.join(CKPT_DIR, "config.json")):
    print(f"✅ Model found at {CKPT_DIR}. Skipping download.")
else:
    print(f"Downloading {REPO_ID} → {CKPT_DIR} (First run takes 10–30 min).")
    # This will resume where it left off if it crashed before
    snapshot_download(
        repo_id=REPO_ID, 
        local_dir=CKPT_DIR, 
        local_dir_use_symlinks=False, 
        resume_download=True
    )
    print("Done.")

In [ ]:
# @title 4. Prompt & settings
# @markdown ### Prompt and generation settings
PROMPT = "A cinematic shot of a fox running through a snowy forest at golden hour, 4k, shallow depth of field"  # @param {type:"string"}
SIZE   = "1280*704"  # @param ["1280*704", "704*1280", "832*480", "480*832"]
SEED   = 42  # @param {type:"integer"}
# @markdown Leave IMAGE empty for pure text-to-video. Provide a path for image-to-video (TI2V-5B only).
IMAGE  = ""  # @param {type:"string"}
print(f"MODEL={MODEL}  SIZE={SIZE}  SEED={SEED}")
print(f"PROMPT={PROMPT}")

In [ ]:
# @title 5. Run inference
import subprocess, time, os
%cd /content/Wan2.2

if MODEL == "TI2V-5B":
    task = "ti2v-5B"
    # L4 / 24 GB friendly — heavy offload
    flags = "--offload_model True --convert_model_dtype --t5_cpu"
else:
    task = "t2v-A14B"
    # A100 40GB — offload keeps it under 40 GB
    flags = "--offload_model True --convert_model_dtype"

img_arg = f'--image "{IMAGE}"' if IMAGE else ""
cmd = (
    f'python generate.py --task {task} --size {SIZE} '
    f'--ckpt_dir "{CKPT_DIR}" --base_seed {SEED} {flags} {img_arg} '
    f'--use_flash_attn false --prompt "{PROMPT}"'
)
print("Running:\n", cmd, "\n")
t0 = time.time()
!{cmd}
print(f"\nElapsed: {(time.time()-t0)/60:.1f} min")

In [ ]:
# @title 6. Show result + save to Drive
import glob, shutil, os, time
from IPython.display import HTML
from base64 import b64encode

vids = sorted(glob.glob("/content/Wan2.2/*.mp4"), key=os.path.getmtime, reverse=True)
assert vids, "No mp4 produced — check the inference cell output above."
latest = vids[0]

# Copy to Drive for safekeeping
ts = time.strftime("%Y%m%d_%H%M%S")
drive_out = f"/content/drive/MyDrive/Wan2.2/outputs/t2v_{MODEL}_{ts}.mp4"
shutil.copy(latest, drive_out)
print(f"Saved: {drive_out}")

data_url = "data:video/mp4;base64," + b64encode(open(latest, 'rb').read()).decode()
HTML(f'<video width=720 controls src="{data_url}"></video>')